# 01 - Generate Synthetic Chart Patterns

The goal here is to build a labeled image dataset entirely from scratch using math.
We define 7 chart pattern classes, generate synthetic OHLC price series for each one,
render them as candlestick chart images, and save them to disk.

**Why synthetic data?**
Real labeled chart data is basically nonexistent — nobody has 10k hand-labeled charts.
So we simulate "ideal" versions of each pattern using Gaussian curves and GBM noise,
then train the model on those. Notebook 03 tests whether the model generalizes to real S&P 500 data.

**Output:** ~10,500 PNG images in `data/synthetic/{train,val}/{class_name}/`

In [3]:
import sys, os, random
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
import io
from tqdm import tqdm
from pathlib import Path

# Save data to /content/ (fast local storage)
# Only checkpoints go to Drive since those are small and need to persist
BASE_DIR      = Path('/content')
SYNTHETIC_DIR = BASE_DIR / 'data' / 'synthetic'
RESULTS_DIR   = BASE_DIR / 'results'

N_PER_CLASS = 1500
N_CANDLES   = 60
IMG_SIZE    = 224
VAL_SPLIT   = 0.15
DATA_SEED   = 42

CLASS_NAMES = [
    'head_and_shoulders',
    'double_top',
    'descending_triangle',
    'inv_head_and_shoulders',
    'double_bottom',
    'ascending_triangle',
    'no_pattern',
]

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)

os.makedirs(RESULTS_DIR / 'figures', exist_ok=True)
for split in ['train', 'val']:
    for name in CLASS_NAMES:
        os.makedirs(SYNTHETIC_DIR / split / name, exist_ok=True)

print(f'Data will be saved to: {SYNTHETIC_DIR}')
print(f'Classes: {CLASS_NAMES}')


Data will be saved to: /content/data/synthetic
Classes: ['head_and_shoulders', 'double_top', 'descending_triangle', 'inv_head_and_shoulders', 'double_bottom', 'ascending_triangle', 'no_pattern']


## Helper Functions

Two small utilities that everything else builds on:

- `add_noise` — sprinkles Gaussian noise onto a price series so patterns don't look too "perfect"
- `prices_to_ohlc` — takes a close-price array and derives Open/High/Low from it using a small random spread

In [4]:
def add_noise(prices, noise_level=0.012):
    noise = np.random.normal(0, noise_level * prices.mean(), size=len(prices))
    return prices + noise


def prices_to_ohlc(closes, spread=0.005):
    n = len(closes)
    ohlc = np.zeros((n, 4))
    for i in range(n):
        c = closes[i]
        o = closes[i-1] if i > 0 else c
        hi = max(o, c) * (1 + abs(np.random.normal(0, spread)))
        lo = min(o, c) * (1 - abs(np.random.normal(0, spread)))
        ohlc[i] = [o, hi, lo, c]
    return ohlc

## Bearish Pattern Generators

We model patterns as sums of Gaussian bumps on a normalized time axis [0, 1].
This gives smooth, recognizable shapes that we then perturb with noise.

- **Head & Shoulders**: three peaks where the middle is tallest, signals trend reversal
- **Double Top**: two roughly equal peaks with a valley in between
- **Descending Triangle**: flat support line, declining resistance — price compressing downward

In [5]:
def gen_head_and_shoulders():
    x = np.linspace(0, 1, N_CANDLES)
    left_shoulder  = 0.70 * np.exp(-((x - 0.20)**2) / 0.003)
    head           = 1.00 * np.exp(-((x - 0.50)**2) / 0.003)
    right_shoulder = 0.65 * np.exp(-((x - 0.80)**2) / 0.003)
    neckline = np.full(N_CANDLES, 0.25)
    prices = 1.0 + left_shoulder + head + right_shoulder + neckline
    return prices_to_ohlc(add_noise(prices))


def gen_double_top():
    x = np.linspace(0, 1, N_CANDLES)
    peak1  =  1.00 * np.exp(-((x - 0.30)**2) / 0.004)
    peak2  =  0.95 * np.exp(-((x - 0.70)**2) / 0.004)
    valley = -0.30 * np.exp(-((x - 0.50)**2) / 0.008)
    prices = 1.0 + peak1 + peak2 + valley
    return prices_to_ohlc(add_noise(prices))


def gen_descending_triangle():
    x = np.linspace(0, 1, N_CANDLES)
    support    = np.ones(N_CANDLES) * 1.0           # flat bottom
    resistance = np.linspace(1.5, 1.05, N_CANDLES) # declining top
    t    = np.linspace(0, 3 * np.pi, N_CANDLES)
    wave = 0.5 * (resistance - support) * (0.5 + 0.5 * np.sin(t)) * np.linspace(1, 0.2, N_CANDLES)
    prices = support + wave
    return prices_to_ohlc(add_noise(prices, noise_level=0.008))

## Bullish Pattern Generators

Bullish patterns are vertical mirrors of their bearish counterparts.
`_flip` inverts the OHLC values around the price midpoint — a Head & Shoulders
flipped upside-down becomes an Inverse Head & Shoulders.

`gen_no_pattern` uses **Geometric Brownian Motion (GBM)**, the standard random walk
model for stock prices — log returns are normally distributed with a drift and volatility term.

In [6]:
def _flip(ohlc):
    hi = ohlc[:, 1].max()
    lo = ohlc[:, 2].min()
    f = ohlc.copy()
    f[:, 0] = hi + lo - ohlc[:, 0]  # open
    f[:, 1] = hi + lo - ohlc[:, 2]  # high becomes old low
    f[:, 2] = hi + lo - ohlc[:, 1]  # low becomes old high
    f[:, 3] = hi + lo - ohlc[:, 3]  # close
    return f


def gen_inv_head_and_shoulders():
    return _flip(gen_head_and_shoulders())

def gen_double_bottom():
    return _flip(gen_double_top())

def gen_ascending_triangle():
    return _flip(gen_descending_triangle())

def gen_no_pattern():
    # GBM: S(t) = S(0) * exp(cumsum of log-normal returns)
    dt = 1 / N_CANDLES
    mu, sigma = 0.0, 0.015
    log_returns = np.random.normal((mu - 0.5 * sigma**2) * dt, sigma * np.sqrt(dt), N_CANDLES)
    prices = 1.0 * np.exp(np.cumsum(log_returns))
    return prices_to_ohlc(prices)


GENERATORS = {
    'head_and_shoulders':     gen_head_and_shoulders,
    'double_top':             gen_double_top,
    'descending_triangle':    gen_descending_triangle,
    'inv_head_and_shoulders': gen_inv_head_and_shoulders,
    'double_bottom':          gen_double_bottom,
    'ascending_triangle':     gen_ascending_triangle,
    'no_pattern':             gen_no_pattern,
}

print("Generators ready:", list(GENERATORS.keys()))

Generators ready: ['head_and_shoulders', 'double_top', 'descending_triangle', 'inv_head_and_shoulders', 'double_bottom', 'ascending_triangle', 'no_pattern']


## Chart Renderer

Each OHLC array gets rendered as a candlestick chart image using matplotlib.
- Green candles: close >= open (price went up)
- Red candles: close < open (price went down)
- Dark background mimics real trading platforms — helps with domain transfer later

We save to a BytesIO buffer and convert to PIL, then resize to 224x224 (standard CNN input size).

In [7]:
def render_chart(ohlc, img_size=IMG_SIZE):
    fig, ax = plt.subplots(figsize=(2.24, 2.24), dpi=100)
    ax.set_facecolor('#0d0d0d')
    fig.patch.set_facecolor('#0d0d0d')

    for i, (o, h, l, c) in enumerate(ohlc):
        color = '#26a69a' if c >= o else '#ef5350'
        # wick
        ax.plot([i, i], [l, h], color=color, linewidth=0.8)
        # body
        body_h = max(abs(c - o), 1e-9)
        ax.add_patch(plt.Rectangle((i - 0.3, min(o, c)), 0.6, body_h, color=color))

    ax.set_xlim(-1, len(ohlc))
    ax.axis('off')
    plt.tight_layout(pad=0)

    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
    plt.close(fig)

    buf.seek(0)
    img = Image.open(buf).convert('RGB').resize((img_size, img_size), Image.LANCZOS)
    return img

In [8]:
# Render one sample of each pattern class to make sure they look right
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for i, (name, gen_fn) in enumerate(GENERATORS.items()):
    ohlc = gen_fn()
    img  = render_chart(ohlc)
    axes[i].imshow(img)
    axes[i].set_title(name.replace('_', '\n'), fontsize=9)
    axes[i].axis('off')

axes[-1].axis('off')  # 7 patterns, 8 slots 
plt.suptitle("One sample per pattern class", fontsize=13, y=1.01)
plt.tight_layout()

os.makedirs(RESULTS_DIR / 'figures', exist_ok=True)
plt.savefig(RESULTS_DIR / 'figures' / 'sample_patterns.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved → results/figures/sample_patterns.png")

Saved → results/figures/sample_patterns.png


## Generate & Save the Full Dataset

`generate_and_save` loops over every class, generates `n` images, and writes them to disk.


In [9]:
def generate_and_save(split='train'):
    n = int(N_PER_CLASS * (1 - VAL_SPLIT)) if split == 'train' else int(N_PER_CLASS * VAL_SPLIT)
    out_root = SYNTHETIC_DIR / split

    for name in CLASS_NAMES:
        os.makedirs(out_root / name, exist_ok=True)

    total = 0
    for name, gen_fn in GENERATORS.items():
        for i in tqdm(range(n), desc=f'{split}/{name}', ncols=80):
            img = render_chart(gen_fn())
            img.save(out_root / name / f'{i:05d}.png')
            total += 1

    print(f'\n{split}: {total} images saved to {out_root}')


generate_and_save('train')
generate_and_save('val')
print("\nAll done!")

train/no_pattern: 100%|█████████████████████| 1275/1275 [02:48<00:00,  7.55it/s]



train: 8925 images saved to /content/data/synthetic/train


val/no_pattern: 100%|█████████████████████████| 225/225 [00:29<00:00,  7.74it/s]


val: 1575 images saved to /content/data/synthetic/val

All done!


In [10]:

counts = {}
for split in ['train', 'val']:
    counts[split] = {}
    for name in CLASS_NAMES:
        folder = SYNTHETIC_DIR / split / name
        counts[split][name] = len(list(folder.glob('*.png')))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, split in zip(axes, ['train', 'val']):
    vals = list(counts[split].values())
    bars = ax.bar(range(len(CLASS_NAMES)), vals, color='steelblue', edgecolor='white', linewidth=0.5)
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels([c.replace('_', '\n') for c in CLASS_NAMES], fontsize=8)
    ax.set_title(f'{split} split')
    ax.set_ylabel('image count')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(v),
                ha='center', va='bottom', fontsize=8)

plt.suptitle("Class Balance Check", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures' / 'class_balance.png', dpi=120, bbox_inches='tight')
plt.show()
print("Dataset generation complete!")
print(f"Train: {sum(counts['train'].values())} images")
print(f"Val  : {sum(counts['val'].values())} images")

Dataset generation complete!
Train: 8925 images
Val  : 1575 images
